# Interior_Design_Render.ipynb — render nội thất / ngoại thất / sân vườn qua API

**Một ảnh vào + prompt → một ảnh render ra.** Notebook mở một HTTP API qua cloudflared
tunnel để gọi từ bên ngoài, kiến trúc lấy từ `ImageVideoToVideo_WanVACE_Colab.ipynb`.

## Notebook này KHÔNG biết taxonomy

Bên ngoài (giao diện hoặc script test) chọn **SPACE → room type → style → color →
keep_layout**, tra 9 file JSON để ghép prompt, rồi **gửi prompt xuống**. Notebook chỉ
nhận chuỗi prompt. Đổi mô tả style/màu là sửa JSON, **không phải sửa và chạy lại notebook**.

| File JSON | Nội dung |
|---|---|
| `space_config_full.json` | 3 nhóm: hậu tố, tag chất lượng, tag hoàn thiện, negative theo nhóm |
| `interior_room_list_full.json` / `exterior_room_list_full.json` | loại phòng + `prompt`, `prompt_must_have`, `negative` |
| `Interior_style_list_full.json` / `exterior_style_list_full.json` / `garden_style_list_full.json` | style + `prompt` (look) + `prompt_color_default` |
| `Interiorcolor_list_full.json` / `enterior_color_list_full.json` / `garden_color_list_full.json` | bảng màu + `prompt` |

Ghép prompt bằng `prompt_builder.py` (cùng thư mục) — cả UI lẫn script test đều dùng chung
một hàm để không bị lệch nhau.

## API

```
POST /generate/interior_design      -> {"job_id": "...", "status": "processing"}
GET  /jobs/{job_id}                 -> tiến độ / lỗi
GET  /jobs/{job_id}/result          -> ảnh JPEG
GET  /version  /health              -> phiên bản, model, tham số
```

Thân JSON tối thiểu: `{"anh_url": "...", "prompt": "..."}`.

## Vì sao job bất đồng bộ

Render 1 ảnh mất 40–120 giây trên T4. Cloudflare cắt kết nối ở **100 giây** (lỗi 524), nên
POST phải trả `job_id` ngay rồi render ở thread nền. `RUN_LOCK` bảo đảm chỉ 1 job chạm GPU
một lúc — T4 16GB không đủ cho 2 pipeline SDXL song song.

> **Bắt buộc:** Runtime > Change runtime type > **GPU (T4)**. Chạy lần lượt cell 1 → 5 và
> **giữ cell 5 chạy** (đóng nó là sập tunnel).


In [ ]:
# ============================================================================
#  Interior_Design_Render.ipynb
#  backend = interior_sdxl_mlsd   (1 notebook = 1 phuong an)
#  Port logic tunnel + job bat dong bo tu ImageVideoToVideo_WanVACE_Colab.ipynb v5
#  Port logic render tu room_redesign_colab.ipynb
# ============================================================================
NOTEBOOK_NAME    = 'Interior_Design_Render.ipynb'
NOTEBOOK_VERSION = 'v3'        # <-- BUMP moi khi sua notebook + cap nhat CHANGELOG
TEST_CASE        = 'v1-sdxl-mlsd-1mp'   #@param {type:"string"}
BACKEND          = 'interior_sdxl_mlsd'

# Checkpoint SDXL. Doi la phai tai lai ~7 GB nen day la lua chon LUC CAI DAT,
# KHONG phai tham so cua job.
BASE_MODEL = 'SG161222/RealVisXL_V5.0'  #@param ["SG161222/RealVisXL_V5.0", "RunDiffusion/Juggernaut-XL-v9", "stabilityai/stable-diffusion-xl-base-1.0"]
# Ban ControlNet rut gon cua diffusers: nhe VRAM hon han, nen bat tren T4 free.
CONTROLNET_SMALL = True  #@param {type:"boolean"}

# Ten type ben api dung de dung duong dan THANG: POST /generate/<type>.
AI_TYPE = 'interior_design'
AI_TYPE_ALIAS = ['interior_render', 'room_redesign', 'thiet_ke_noi_that']

MODELS = [
    {
        "name": "SDXL + ControlNet (MLSD line map) + VAE fp16-fix",
        "code": "https://github.com/huggingface/diffusers",
        "weights": "BASE_MODEL chon o tren + diffusers/controlnet-canny-sdxl-1.0(-small) "
                   "+ madebyollin/sdxl-vae-fp16-fix + lllyasviel/Annotators (MLSD ~50 MB)",
        "license": "SDXL: CreativeML Open RAIL++-M. Checkpoint cong dong: xem giay phep goc.",
        "size": "SDXL ~7 GB + ControlNet 0,7 GB (small) hoac 2,5 GB (full) + VAE 0,3 GB"
    }
]

# Tham so job cong bo qua GET /version. Client chi can gui 'anh_url' + 'prompt'.
BACKEND_PARAMS = ('anh_url', 'anh_base64', 'prompt', 'prompt_2', 'negative_prompt',
                  'negative_prompt_2', 'keep_layout', 'line_scale', 'mlsd_threshold',
                  'guidance', 'steps', 'seed', 'megapixel')

# Do tren room_redesign_colab.ipynb. Doi la phai do lai.
MAC_DINH = {
    # keep_layout=True -> giu cua so / duong tuong cua anh goc; False -> cho doi bo cuc.
    # Hai muc nay la weight cua ControlNet tren ban do duong MLSD.
    "line_scale_keep": 0.60,
    "line_scale_free": 0.25,
    # Nguong nhan duong thang cua MLSD. Cao hon = it duong hon = xoa do cu sach hon,
    # nhung qua cao thi mat ca duong kien truc va phong bat dau meo.
    "mlsd_threshold": 0.10,
    # RealVisXL / Juggernaut thich CFG 4-7. Day len 10-12 la anh chay mau, vien gat.
    "guidance": 6.0,
    "steps": 30,
    "seed": 42,
    # SDXL duoc train quanh 1 megapixel. Anh LUON giu dung ti le goc, chi scale ve muc nay.
    "megapixel": 1.0,
    # Dung khi client khong gui. Prompt that do client ghep tu JSON.
    "prompt": "living room interior, large sofa, coffee table, armchair, area rug, "
              "floor lamp, framed wall art, curtains",
    "prompt_2": "living room, fully furnished, professionally staged, all essential furniture "
                "present, photorealistic, professional interior photography, sharp focus",
    "negative_prompt": "blurry, low quality, distorted, deformed furniture, watermark, text, "
                       "unrealistic proportions, warped walls, people, duplicate objects, "
                       "floating furniture, oversaturated",
    "negative_prompt_2": "empty room, unfurnished, bare floor, no furniture, vacant",
}

CHANGELOG = {
    "v3": "Do that v2 van bi ngat sau vai job (xem v2). empty_cache()/ipc_collect() moi "
          "job chi tra duoc khoi RANH cho allocator, KHONG gom lai duoc khoi da phan manh -- "
          "nen 'da don' nhin sach (memory_allocated thap) ma phien van chet dan. Sua 3 cho: "
          "(1) dat os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True' TRUOC "
          "dong `import torch` dau tien, giam phan manh o tang allocator; "
          "(2) _moc()/LAST_STATS gio ghi CA memory_reserved() ben canh memory_allocated() -- "
          "so nay moi lo ro phan manh, vi no la tong pool allocator dang giu ke ca phan "
          "khong tra duoc; (3) them RELOAD_EVERY (mac dinh 8): sau tung nay job thi xoa het "
          "PIPE va goi lai nap_model() de reset sach allocator, thay vi tin empty_cache() du "
          "dung sau moi job. CHUA DO LAI tren cum that -- can theo doi vram_reserved_sau_don_gb "
          "qua nhieu job de xac nhan no khong tang dan nua, va xac nhan phien khong bi ngat "
          "som hon truoc khi cham RELOAD_EVERY.",
    "v2": "Don bo nho sau MOI job (gc.collect + torch.cuda.empty_cache + ipc_collect). "
          "DO THAT tren cum 17/09/2026, cung mot may Colab, 5 job Garden lien tiep: "
          "73s -> 186s -> 314s -> 351s -> 320s roi phien bi ngat. Job thu tu cham gap "
          "gan 5 lan job dau. Nguyen nhan: pipeline nam thuong tru trong PIPE nhung cache "
          "cua CUDA allocator khong duoc tra lai, va enable_model_cpu_offload day model "
          "qua lai RAM/VRAM moi job nen phan manh don. "
          "CHUA DO LAI sau khi sua — can chay lai >=5 job de xac nhan thoi gian khong tang dan.",
    "v1": "Ban dau. Tunnel + job bat dong bo + callback port tu WanVACE v5. Render port tu "
          "room_redesign_colab.ipynb: SDXL + ControlNet tren ban do duong MLSD, VAE fp16-fix "
          "(VAE goc cua SDXL ra NaN khi chay fp16 tren T4 -> anh den si). "
          "KHAC voi room_redesign: notebook KHONG biet taxonomy, chi nhan prompt tu client. "
          "CHUA DO: thoi gian that moi job tren T4 (uoc 40-120 s tuy steps va megapixel)."
}
print(f'{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | base={BASE_MODEL}')


In [ ]:
import os, shutil, subprocess, sys, time

_T0 = time.time()


def _run(cmd, check=True, quiet=True):
    """Chay lenh, in dong lenh truoc de log Colab doc duoc dang o buoc nao."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=quiet, text=True)
    if check and r.returncode != 0:
        duoi = (r.stderr or r.stdout or '').strip()[-1200:]
        raise RuntimeError(f'lenh loi ({r.returncode}): {cmd}\n--- stderr ---\n{duoi}')
    return r


def _pip(*pk):
    _run([sys.executable, '-m', 'pip', 'install', '-q', *pk])


# cloudflared: cell server goi thang /usr/local/bin/cloudflared va KIEM LAI ngay sau khi
# cai. Cai hong ma khong kiem thi model nap xong 5 phut moi vo o dong cuoi.
if not shutil.which('cloudflared'):
    _run('curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/'
         'releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared')
if not shutil.which('cloudflared'):
    raise RuntimeError('khong cai duoc cloudflared -- cell server se khong mo duoc tunnel')

_pip('--upgrade', 'diffusers', 'transformers', 'accelerate', 'safetensors', 'peft')
_pip('controlnet_aux', 'opencv-python-headless', 'fastapi', 'uvicorn', 'python-multipart',
     'requests', 'psutil', 'pillow')

print(f'>>> cai dat xong trong {time.time() - _T0:.0f}s')


In [ ]:
import base64, io as _io, json, os, re, time
from typing import Optional

import requests
from pydantic import BaseModel

JOB_DIR, RES_DIR = '/content/jobs', '/content/results'
for d in (JOB_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

LAST_STATS = {}


def _slug(s, default='case'):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', (s or '').strip()).strip('-.')
    return (s or default)[:48]


def _moc(ten):
    """Moc bo nho tai tung buoc. T4 chet o RAM 12,7 GB nhieu khong kem gi VRAM,
    nen phai thay duoc RAM tang o buoc nao.

    Theo CA memory_allocated() (tensor dang song) LAN memory_reserved() (tong pool
    ma CUDA allocator dang giu, ke ca phan da phan manh chua tra duoc). empty_cache()
    co the lam memory_allocated() nhin sach trong khi memory_reserved() van cao -- chi
    nhin memory_allocated() (nhu ban v2) se KHONG thay duoc phan manh dang tich luy.
    """
    try:
        import psutil
        ram = round(psutil.Process(os.getpid()).memory_info().rss / 1024 ** 3, 2)
    except Exception:
        ram = None
    try:
        import torch
        vram = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) if torch.cuda.is_available() else None
        vram_reserved = round(torch.cuda.memory_reserved() / 1024 ** 3, 2) if torch.cuda.is_available() else None
    except Exception:
        vram = vram_reserved = None
    LAST_STATS.setdefault('_bo_nho', {})[ten] = {'ram_gb': ram, 'vram_gb': vram,
                                                 'vram_reserved_gb': vram_reserved,
                                                 't': round(time.time(), 1)}


class Job(BaseModel):
    """Than JSON cua POST /generate/interior_design.

    Notebook KHONG biet SPACE / room type / style / color. Client tra 9 file JSON,
    ghep chuoi roi gui xuong day. Xem prompt_builder.py.

    anh_url / anh_base64: anh phong goc. Uu tien anh_url; anh_base64 cho client khong
                          co cho host file (data URI hoac base64 tran deu nhan).
    prompt              : encoder 1 - khong gian + hang muc + look cua style + bang mau
    prompt_2            : encoder 2 - hang muc dinh danh + tag hoan thien + tag chat luong
                          SDXL co 2 text encoder, moi cai chi nhan 77 token. Tach lam doi
                          de khong bi cat mat quality tag o cuoi.
    keep_layout         : True  -> giu cua so / duong tuong cua anh goc (line_scale 0.60)
                          False -> cho model doi bo cuc (line_scale 0.25)
    """
    anh_url: Optional[str] = None
    anh_base64: Optional[str] = None
    prompt: Optional[str] = None
    prompt_2: Optional[str] = None
    negative_prompt: Optional[str] = None
    negative_prompt_2: Optional[str] = None
    keep_layout: bool = True
    # None = de MAC_DINH quyet.
    line_scale: Optional[float] = None      # ghi de line_scale_keep/free neu muon chinh tay
    mlsd_threshold: Optional[float] = None
    guidance: Optional[float] = None
    steps: Optional[int] = None
    seed: Optional[int] = None
    megapixel: Optional[float] = None
    test_case: str = ''
    job_id: Optional[str] = None
    callback_url: Optional[str] = None
    callback_token: Optional[str] = None


def tai_anh(job: Job, uid: str) -> str:
    """Lay anh goc ve dia. Tra duong dan file."""
    if job.anh_base64:
        raw = job.anh_base64.split(',', 1)[-1]      # bo tien to data:image/...;base64,
        duong = os.path.join(JOB_DIR, f'{uid}_in.png')
        with open(duong, 'wb') as f:
            f.write(base64.b64decode(raw))
        return duong
    if not job.anh_url:
        raise ValueError('thieu anh dau vao: can anh_url hoac anh_base64')
    r = requests.get(job.anh_url, stream=True, timeout=120,
                     headers={'User-Agent': 'interior-render/1.0'})
    r.raise_for_status()
    duoi = os.path.splitext(job.anh_url.split('?')[0])[1].lower()
    if duoi not in ('.jpg', '.jpeg', '.png', '.webp', '.bmp'):
        duoi = '.jpg'
    duong = os.path.join(JOB_DIR, f'{uid}_in{duoi}')
    with open(duong, 'wb') as f:
        for chunk in r.iter_content(1 << 16):
            f.write(chunk)
    if os.path.getsize(duong) < 1024:
        raise ValueError(f'anh tai ve qua nho ({os.path.getsize(duong)} byte) - URL hong?')
    return duong


print('Job model + helper san sang. Thu muc:', JOB_DIR, RES_DIR)


In [ ]:
import gc, os, time

# Giam phan manh cua CUDA allocator khi enable_model_cpu_offload day model qua lai
# RAM/VRAM MOI JOB (xem CHANGELOG v2: 5 job Garden lien tiep 73s -> 186s -> 314s ->
# 351s -> 320s roi phien bi ngat, du da goi empty_cache()/ipc_collect() sau moi job).
# PHAI dat truoc dong `import torch` dau tien trong ca notebook, neu khong bien nay
# khong co tac dung (allocator doc gia tri nay luc khoi tao).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import cv2
import numpy as np
import torch
from PIL import Image
from diffusers import (StableDiffusionXLControlNetPipeline, ControlNetModel,
                       AutoencoderKL, UniPCMultistepScheduler)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
PIPE = {'pipe': None, 'mlsd': None}

# empty_cache()/ipc_collect() sau moi job KHONG du de tra het phan manh (do that
# 17/09/2026, xem CHANGELOG). Nap lai TOAN BO pipeline dinh ky de reset sach trang
# thai allocator, thay vi cho phan manh tich luy den khi Colab tu ngat phien.
RELOAD_EVERY = 8          #@param {type:"integer"}  0 = tat, khong nap lai dinh ky
JOB_COUNT = {'n': 0}

CONTROLNET_ID = ('diffusers/controlnet-canny-sdxl-1.0-small' if CONTROLNET_SMALL
                 else 'diffusers/controlnet-canny-sdxl-1.0')


def nap_model():
    """Chay o PRELOAD. Tai ~8 GB o lan dau, sau do nam trong cache HF cua phien."""
    if PIPE['pipe'] is not None:
        return
    if DEVICE == 'cpu':
        raise RuntimeError('khong thay GPU - Runtime > Change runtime type > GPU')

    from controlnet_aux import MLSDdetector
    _moc('tai_mlsd')
    print(f'   [{BACKEND}] tai MLSD line detector ...', flush=True)
    PIPE['mlsd'] = MLSDdetector.from_pretrained('lllyasviel/Annotators')

    _moc('tai_controlnet')
    print(f'   [{BACKEND}] tai ControlNet {CONTROLNET_ID} ...', flush=True)
    controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, torch_dtype=DTYPE)

    # VAE goc cua SDXL tran so khi chay fp16 tren T4 -> anh ra den si. Ban fp16-fix
    # la bat buoc, khong phai toi uu.
    _moc('tai_vae')
    vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)

    _moc('tai_sdxl')
    print(f'   [{BACKEND}] tai SDXL {BASE_MODEL} ...', flush=True)

    def _tai(variant):
        return StableDiffusionXLControlNetPipeline.from_pretrained(
            BASE_MODEL, controlnet=controlnet, vae=vae, torch_dtype=DTYPE,
            use_safetensors=True, variant=variant)

    try:
        pipe = _tai('fp16' if DTYPE == torch.float16 else None)
    except Exception as e:
        # Nhieu checkpoint cong dong khong up ban fp16 rieng.
        print(f'   khong co variant fp16 ({type(e).__name__}), tai ban day du ...')
        pipe = _tai(None)

    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_model_cpu_offload()     # T4 16GB: offload tung module khi khong dung
    # diffusers moi bo 2 method nay o cap pipeline, chuyen xuong pipe.vae
    for _n in ('enable_slicing', 'enable_tiling'):
        if hasattr(pipe.vae, _n):
            getattr(pipe.vae, _n)()
    PIPE['pipe'] = pipe
    gc.collect()
    torch.cuda.empty_cache()
    _moc('model_san_sang')
    print(f'   [{BACKEND}] model san sang tren {torch.cuda.get_device_name(0)}')


def _nap_lai_dinh_ky():
    """Xoa het pipeline hien tai roi nap lai tu dau -- reset sach trang thai CUDA
    allocator. empty_cache()/ipc_collect() moi job chi tra duoc khoi RANH, phan da
    phan manh (khong con khoi lien tuc du lon) thi khong tra duoc; nap lai la cach
    chac chan duy nhat de dua allocator ve trang thai sach nhu luc moi khoi dong."""
    print(f'   [{BACKEND}] da render {JOB_COUNT["n"]} job -- nap lai pipeline de '
          f'xoa phan manh allocator (RELOAD_EVERY={RELOAD_EVERY}) ...', flush=True)
    PIPE['pipe'] = None
    PIPE['mlsd'] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    nap_model()


def fit_size(w, h, megapixel=1.0):
    """Giu dung ti le goc, scale ve ~megapixel, lam tron boi so 8 (yeu cau cua SDXL).

    KHONG bop ve anh vuong: anh 16:9 bi ep thanh 1:1 la tuong va cua so meo het.
    """
    ar = w / h
    target = megapixel * 1024 * 1024
    nh = (target / ar) ** 0.5
    nw = ar * nh
    return max(512, int(round(nw / 8) * 8)), max(512, int(round(nh / 8) * 8))


def make_mlsd(image, thr=0.1):
    """Ban do duong THANG DAI -> gan nhu chi con kien truc.

    Dung MLSD chu khong phai Canny/Depth: Canny bat MOI duong bien nen giu ca duong
    vien do dac cu (model se nhoi do moi vao dung hinh do cu, ra do meo); Depth con
    giu nguyen khoi 3D cua do cu. MLSD chi bat duong thang dai = chan tuong, khung
    cua so, goc tuong -> xoa do cu roi bay lai tu dau moi sach.
    """
    out = PIPE['mlsd'](image, thr_v=thr, thr_d=0.1,
                       detect_resolution=512, image_resolution=max(image.size))
    return out.convert('RGB').resize(image.size, Image.LANCZOS)


def chay_job(job: Job, uid: str, out_path: str, progress=None):
    """Render 1 anh. Tra duong dan file ket qua."""
    nap_model()
    t0 = time.time()
    cfg = {k: (getattr(job, k) if getattr(job, k, None) is not None else MAC_DINH.get(k))
           for k in ('mlsd_threshold', 'guidance', 'steps', 'seed', 'megapixel')}
    line_scale = job.line_scale
    if line_scale is None:
        line_scale = MAC_DINH['line_scale_keep'] if job.keep_layout else MAC_DINH['line_scale_free']

    if progress:
        progress('tai_anh', 0, 3)
    duong_vao = tai_anh(job, uid)

    src = Image.open(duong_vao).convert('RGB')
    size = fit_size(*src.size, megapixel=cfg['megapixel'])
    anh = src.resize(size, Image.LANCZOS)

    if progress:
        progress('mlsd', 1, 3)
    _moc('mlsd')
    line = make_mlsd(anh, thr=cfg['mlsd_threshold'])

    if progress:
        progress('denoise', 2, 3)
    _moc('truoc_denoise')
    pipe = PIPE['pipe']
    # Noi tien do tung buoc denoise cua diffusers vao progress cua job.
    def _cb(p, step, timestep, kw):
        if progress:
            progress('denoise', step + 1, cfg['steps'])
        return kw

    ket_qua = pipe(
        prompt=job.prompt or MAC_DINH['prompt'],
        prompt_2=job.prompt_2 or MAC_DINH['prompt_2'],
        negative_prompt=job.negative_prompt or MAC_DINH['negative_prompt'],
        negative_prompt_2=job.negative_prompt_2 or MAC_DINH['negative_prompt_2'],
        image=line,
        controlnet_conditioning_scale=float(line_scale),
        num_inference_steps=int(cfg['steps']),
        guidance_scale=float(cfg['guidance']),
        width=size[0], height=size[1],
        generator=torch.Generator(device='cpu').manual_seed(int(cfg['seed'])),
        callback_on_step_end=_cb,
    ).images[0]

    # ComfyUI/diffusers KHONG bao khi tensor ra rac: kiem o day, khong thi job bao
    # 'done' ma anh den si.
    arr = np.asarray(ket_qua)
    if arr.std() < 0.5:
        raise RuntimeError(f'anh ra RONG (do lech chuan {arr.std():.3f}/255) - VAE tran so?')

    ket_qua.save(out_path, quality=95)
    _moc('xong')
    LAST_STATS.clear()
    LAST_STATS.update({
        # Version di kem TUNG JOB: so_lieu la duong duy nhat tu Colab ve client qua api,
        # khong co no thi khong biet may dang chay ban nao.
        'notebook_version': NOTEBOOK_VERSION, 'backend': BACKEND,
        'giay': round(time.time() - t0, 1), 'kich_thuoc': list(size),
        'line_scale': float(line_scale), 'keep_layout': job.keep_layout,
        'steps': cfg['steps'], 'guidance': cfg['guidance'], 'seed': cfg['seed'],
        'mlsd_threshold': cfg['mlsd_threshold'],
        'prompt': (job.prompt or MAC_DINH['prompt'])[:400],
    })
    # DON SAU MOI JOB. Pipeline nam thuong tru trong PIPE (dung y do: nap lai mat 5
    # phut), nhung cache cua CUDA allocator thi khong duoc giu. Khong don thi do that
    # tren cum 17/09/2026 la: 73s -> 186s -> 314s -> 351s roi Colab bi ngat.
    # empty_cache() tra khoi da cap phat ve driver; ipc_collect() don handle chia se.
    del ket_qua, arr, line, anh, src
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    _moc('da_don')
    LAST_STATS['vram_sau_don_gb'] = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) \
        if torch.cuda.is_available() else None
    # reserved > allocated sau don la dau hieu phan manh: allocator van giu khoi do
    # trong tay (khong tra ve driver) du khong con tensor nao dung no.
    LAST_STATS['vram_reserved_sau_don_gb'] = round(torch.cuda.memory_reserved() / 1024 ** 3, 2) \
        if torch.cuda.is_available() else None

    print(f'   [{BACKEND}] xong {size[0]}x{size[1]} trong {LAST_STATS["giay"]}s '
          f'(vram con {LAST_STATS["vram_sau_don_gb"]} GB, '
          f'reserved {LAST_STATS["vram_reserved_sau_don_gb"]} GB)')

    JOB_COUNT['n'] += 1
    if RELOAD_EVERY and JOB_COUNT['n'] % RELOAD_EVERY == 0:
        _nap_lai_dinh_ky()

    return out_path


PRELOAD = [('nap_model', nap_model)]
print('Ham render san sang. PRELOAD se nap model o cell server.')


In [ ]:
import os, re, socket, subprocess, threading, time, traceback, uuid

import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, HTMLResponse

PORT = 8000
STATE = {'models_ready': False}
app = FastAPI(title=f'{NOTEBOOK_NAME} [{BACKEND}] {NOTEBOOK_VERSION}')

# ============================================================ job store (async)
# Tra job_id ngay roi render o thread nen -> tranh Cloudflare 524 (cat o 100 giay)
# trong khi 1 anh mat 40-120 giay.
JOBS, JOBS_LOCK, RUN_LOCK, JOB_TTL = {}, threading.Lock(), threading.Lock(), 6 * 3600


def _prune_locked():
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('result') and os.path.exists(v['result']):
                os.remove(v['result'])
        except Exception:
            pass


def _new_job(jid=None, callback_url=None, callback_token=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_locked()
        JOBS[jid] = {'status': 'processing', 'stage': 'queued', 'progress': 0.0,
                     'result': None, 'filename': None, 'error': None, 'created': time.time(),
                     'callback_url': callback_url, 'callback_token': callback_token,
                     'da_day_ve': None}
    return jid


def _progress_cb(jid):
    def cb(stage, done, total):
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['stage'] = stage
                JOBS[jid]['progress'] = round(done / max(1, total), 4)
    return cb


def day_ve_server(jid, duong):
    """Colab tu day ket qua ve api.

    Do tren chinh ha tang nay: quick tunnel cloudflared cho chieu api-KEO khoang
    0,11 MB/s, chieu Colab-DAY khoang 8,5 MB/s. Khong co callback_url thi KHONG lam
    gi: api keo qua GET /jobs/{id}/result nhu cu.
    """
    with JOBS_LOCK:
        j = JOBS.get(jid) or {}
        url, token = j.get('callback_url'), j.get('callback_token')
    if not url or not token or not duong or not os.path.exists(duong):
        return None
    mb = os.path.getsize(duong) / 1e6
    t0 = time.perf_counter()
    try:
        with open(duong, 'rb') as f:
            r = requests.post(url, data={'token': token},
                              files={'file': (os.path.basename(duong), f, 'image/jpeg')},
                              timeout=(15, 900))
        if r.status_code != 200:
            print(f'[{BACKEND}] day ve server that bai HTTP {r.status_code}: {r.text[:200]}')
            return False
        giay = time.perf_counter() - t0
        print(f'[{BACKEND}] da day {mb:.2f} MB ve server trong {giay:.1f}s')
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['da_day_ve'] = True
        return True
    except Exception as e:
        print(f'[{BACKEND}] day ve server loi: {e}')
        return False


def _run_async(jid, tag, work):
    try:
        with RUN_LOCK:                      # serialize GPU: chi 1 job render 1 luc
            path, filename = work()
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='done', stage='done', progress=1.0,
                                 result=path, filename=filename, so_lieu=dict(LAST_STATS))
        day_ve_server(jid, path)
        print(f'[{tag}] xong job {jid[:8]} -> {path}')
    except Exception as e:
        traceback.print_exc()
        tb = traceback.format_exc()
        frames = [ln for ln in tb.splitlines() if ln.strip().startswith('File "')]
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='error', stage='error', error=str(e)[:800],
                                 error_type=type(e).__name__,
                                 where=frames[-1].strip()[:300] if frames else None,
                                 traceback=tb[-2500:])
        print(f'[{tag}] LOI job {jid[:8]}: {type(e).__name__}: {e}')


def _spawn(jid, tag, work, test_case=None):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid]['test_case'] = _slug(test_case or TEST_CASE)
    print(f'[{tag}] nhan job {jid[:8]} case={_slug(test_case or TEST_CASE)} -> chay nen.')
    threading.Thread(target=_run_async, args=(jid, tag, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing', 'backend': BACKEND,
            'test_case': _slug(test_case or TEST_CASE),
            'result_url': f'/jobs/{jid}/result',
            'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION}


def _active_jobs():
    with JOBS_LOCK:
        return sum(1 for v in JOBS.values() if v.get('status') == 'processing')


def _out_paths(uid, test_case, ext='jpg'):
    fn = f'{uid}_{BACKEND}_{_slug(test_case)}_{NOTEBOOK_VERSION}.{ext}'
    return os.path.join(RES_DIR, fn), fn


STARTED_AT = time.time()


def _json_an_toan(o):
    """Starlette dump JSON voi allow_nan=False -- mot NaN lot vao la endpoint tra HTTP 500."""
    import math
    if isinstance(o, float):
        return o if math.isfinite(o) else str(o)
    if isinstance(o, dict):
        return {k: _json_an_toan(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_an_toan(v) for v in o]
    return o


# ============================================================ endpoints
@app.get('/version')
def version():
    try:
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        gpu = None
    return {'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION, 'backend': BACKEND,
            'ai_type': AI_TYPE, 'generate_paths': DUONG_GENERATE, 'test_case': TEST_CASE,
            'base_model': BASE_MODEL, 'controlnet': CONTROLNET_ID,
            'models': MODELS, 'changelog': CHANGELOG, 'params': list(BACKEND_PARAMS),
            'mac_dinh': MAC_DINH, 'gpu': gpu, 'models_ready': STATE['models_ready'],
            'uptime_s': round(time.time() - STARTED_AT, 1),
            'jobs': {'total': len(JOBS), 'active': _active_jobs(),
                     'done': sum(1 for v in JOBS.values() if v.get('status') == 'done'),
                     'error': sum(1 for v in JOBS.values() if v.get('status') == 'error')}}


@app.get('/health')
def health():
    return {'ok': True, 'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION,
            'backend': BACKEND, 'ai_type': AI_TYPE, 'test_case': TEST_CASE,
            'generate_paths': DUONG_GENERATE, 'models_ready': STATE['models_ready'],
            'active_jobs': _active_jobs(), 'params': list(BACKEND_PARAMS)}


@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        return _json_an_toan({
            'job_id': job_id, 'status': j['status'], 'stage': j['stage'],
            'progress': j['progress'], 'error': j['error'],
            'elapsed': round(time.time() - j['created'], 1),
            'backend': BACKEND, 'notebook_version': NOTEBOOK_VERSION,
            'test_case': j.get('test_case'), 'filename': j.get('filename'),
            'so_lieu': j.get('so_lieu'), 'da_day_ve': j.get('da_day_ve'),
            'error_type': j.get('error_type'), 'where': j.get('where'),
            'traceback': j.get('traceback')})


@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        status, path, fn, err = j['status'], j['result'], j['filename'], j['error']
    if status == 'processing':
        raise HTTPException(status_code=409, detail='job chua xong')
    if status == 'error':
        raise HTTPException(status_code=500, detail=err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=410, detail='ket qua khong con (da bi don)')
    return FileResponse(path, media_type='image/jpeg', filename=fn)


@app.get('/laststats')
def laststats():
    return _json_an_toan(LAST_STATS)


@app.get('/', response_class=HTMLResponse)
def trang_chu():
    return (f'<!doctype html><meta charset=utf-8><title>{NOTEBOOK_NAME}</title>'
            f'<body style="font:14px system-ui;max-width:680px;margin:2rem auto">'
            f'<h3>{NOTEBOOK_NAME} {NOTEBOOK_VERSION} &mdash; backend <code>{BACKEND}</code></h3>'
            f'<p><a href=/version>/version</a> &middot; <a href=/health>/health</a> &middot; '
            f'<a href=/laststats>/laststats</a></p>'
            f'<p>POST JSON toi <code>{DUONG_GENERATE[0]}</code> voi '
            f'<code>{{"anh_url": "...", "prompt": "..."}}</code>, roi GET /jobs/{{id}} va '
            f'/jobs/{{id}}/result.</p>'
            f'<p>Notebook KHONG biet SPACE/room/style/color &mdash; client ghep prompt tu '
            f'cac file JSON roi gui xuong.</p>')


def nhan_job(job: Job):
    uid = uuid.uuid4().hex[:8]
    case = job.test_case or TEST_CASE
    out_path, fn = _out_paths(uid, case, ext='jpg')
    jid = _new_job(job.job_id, job.callback_url, job.callback_token)
    prog = _progress_cb(jid)

    def work():
        return chay_job(job, uid, out_path, progress=prog), fn

    return _spawn(jid, BACKEND, work, case)


# api dung duong dan THANG tu ten type: POST /generate/<type>. Dang ky CUNG MOT ham
# duoi moi ten; dict.fromkeys de giu thu tu va bo trung.
DUONG_GENERATE = ['/generate/' + t for t in dict.fromkeys([AI_TYPE] + AI_TYPE_ALIAS)]
for _d in DUONG_GENERATE:
    app.post(_d)(nhan_job)


# ==== KHOI DONG ====
print(f'>>> [{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] backend={BACKEND} case={TEST_CASE}')
_ok = True
for _name, _fn in PRELOAD:
    _t0 = time.time()
    try:
        _fn()
        print(f'   [preload] {_name} OK ({time.time() - _t0:.0f}s)')
    except Exception as _e:
        _ok = False
        traceback.print_exc()
        print(f'   [preload] {_name} LOI: {_e}')
STATE['models_ready'] = _ok
print('>>> MODEL SAN SANG.' if _ok else '>>> MODEL CHUA SAN SANG (xem log tren).')


def _port_in_use(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', p)) == 0


if _port_in_use(PORT):
    print(f'uvicorn da chay san o port {PORT} (chay lai cell) -> KHONG khoi dong lai.')
else:
    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT,
                                                log_level='warning'), daemon=True).start()
    time.sleep(3)

try:
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
except Exception:
    pass

# Bao PUBLIC_URL len api: bot bom window.__ai_boot truoc khi mo notebook.
AI_API_BASE = 'https://api.downloadvideo.vn'


def _read_ai_boot(tries=6, wait=5):
    import json as _json
    try:
        from google.colab import output as _out
    except Exception as e:
        print('   [report] khong co google.colab.output:', e)
        return None
    for i in range(1, tries + 1):
        try:
            raw = _out.eval_js('JSON.stringify(window.__ai_boot || null)', timeout_sec=20)
            boot = _json.loads(raw) if raw and raw != 'null' else None
            if boot and boot.get('account') and boot.get('token'):
                return boot
            print(f'   [report] lan {i}/{tries}: chua thay window.__ai_boot')
        except Exception as e:
            print(f'   [report] lan {i}/{tries}: eval_js loi: {e}')
        time.sleep(wait)
    return None


def _report_public_url(url):
    boot = _read_ai_boot()
    if not boot:
        print('   [report] KHONG co __ai_boot (chay tay?) -> bot se quet dong PUBLIC_URL=.')
        return False
    api = (boot.get('api') or AI_API_BASE).rstrip('/')
    body = {'account': boot['account'], 'token': boot['token'], 'url': url}
    hdr = {'ngrok-skip-browser-warning': '1', 'User-Agent': 'colab-tunnel-report/1.0'}
    for attempt in range(1, 25):
        try:
            r = requests.post(f'{api}/api/c/ai/profiles/tunnel', json=body, headers=hdr, timeout=20)
            print(f'   [report] {attempt}/24 -> {r.status_code} {r.text[:200]}')
            if r.status_code == 200 and r.json().get('success'):
                return True
            if r.status_code in (400, 401):
                return False
        except Exception as e:
            print(f'   [report] {attempt}/24 loi mang: {e}')
        time.sleep(15)
    return False


proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

if public_url:
    threading.Thread(target=_report_public_url, args=(public_url,), daemon=True).start()

print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
print('URL:', public_url)
print('Endpoints: GET /version | GET /health | POST ' + DUONG_GENERATE[0] + ' (JSON)')
print('           GET /jobs/{id} | GET /jobs/{id}/result | GET /laststats')
print('>>> SAN SANG. Giu cell nay chay.')
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')
